# Project Name: Nykaa-Inspired Beauty Retail Analytics Dashboard

**Made by:** Madhulika Wakalkar

**Tools used:** Python (Pandas, NumPy, Matplotlib, Seaborn), SQL (SQLite), Power BI

**Goal:** Analyze a real Nykaa product catalog (`dim_products`) paired with a synthetic transactions table (`fact_orders`) to answer business questions like top-performing categories, brand revenue contribution, and monthly sales trends — then export the results for a Power BI dashboard.


## Step 1: Importing Important Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import random
from datetime import datetime, timedelta

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)


## Step 2: Loading the Data (`dim_products`)

This is the real Kaggle product catalog. It describes each product but has no sales/order info — that's why we'll build `fact_orders` separately in Step 5.

In [7]:
# Update this path to match your local repo structure
df = pd.read_csv(r"C:\Users\praful\Desktop\madhulika\nykaa beauty retail analytics\data\raw\nyka_popular_brands_products_2022_10_16.csv")
print(df.shape)
df.head()


(3663, 14)


,brand_name,product_id,image_url,in_stock,mrp,price,product_title,rating,rating_count,tags,product_url,listing_page_name,listing_url,listing_page_no
0,Herbal Essences,2659739,https://images-static.nykaa.com/media/catalog/...,True,1250,750,Herbal Essences Argan Oil Of Moroccan Shampoo ...,4.4,1008,"FEATURED, BESTSELLER",https://www.nykaa.com/herbal-essences-argan-oi...,Herbal Essences,https://www.nykaa.com/brands/herbal-essences/c...,1
1,Herbal Essences,1290145,https://images-static.nykaa.com/media/catalog/...,True,1575,1181,Herbal Essences Aloe & Bamboo Shampoo + Condit...,4.4,1034,FEATURED,https://www.nykaa.com/herbal-essences-potent-a...,Herbal Essences,https://www.nykaa.com/brands/herbal-essences/c...,1
2,Herbal Essences,456559,https://images-static.nykaa.com/media/catalog/...,True,1250,688,Herbal Essences Argan Oil Shampoo & Conditione...,4.3,10879,"FEATURED, BESTSELLER",https://www.nykaa.com/herbal-essences-argan-oi...,Herbal Essences,https://www.nykaa.com/brands/herbal-essences/c...,1
3,Herbal Essences,3753166,https://images-static.nykaa.com/media/catalog/...,True,1575,945,Herbal Essences Aloe & Bamboo Shampoo + Condit...,4.3,65,FEATURED,https://www.nykaa.com/herbal-essences-soha-alo...,Herbal Essences,https://www.nykaa.com/brands/herbal-essences/c...,1
4,Herbal Essences,5360837,https://images-static.nykaa.com/media/catalog/...,True,600,390,Herbal Essences Argan Oil Of Morocco Shampoo -...,4.3,8769,"FEATURED, BESTSELLER",https://www.nykaa.com/herbal-essences-argan-oi...,Herbal Essences,https://www.nykaa.com/brands/herbal-essences/c...,1


## Step 3: Initial Data Understanding

In [8]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3663 entries, 0 to 3662
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   brand_name         3663 non-null   object 
 1   product_id         3663 non-null   int64  
 2   image_url          3663 non-null   object 
 3   in_stock           3363 non-null   object 
 4   mrp                3663 non-null   int64  
 5   price              3663 non-null   int64  
 6   product_title      3663 non-null   object 
 7   rating             3663 non-null   float64
 8   rating_count       3663 non-null   int64  
 9   tags               305 non-null    object 
 10  product_url        3663 non-null   object 
 11  listing_page_name  3663 non-null   object 
 12  listing_url        3663 non-null   object 
 13  listing_page_no    3663 non-null   int64  
dtypes: float64(1), int64(5), object(8)
memory usage: 400.8+ KB


In [9]:
df.isnull().sum()


brand_name              0
product_id              0
image_url               0
in_stock              300
mrp                     0
price                   0
product_title           0
rating                  0
rating_count            0
tags                 3358
product_url             0
listing_page_name       0
listing_url             0
listing_page_no         0
dtype: int64

In [10]:
df.describe(include='all').T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
brand_name,3663,25,Nykaa Cosmetics,812,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_id,3663.0,NaN,NaN,NaN,1945670.493584,2171660.723064,250.0,333164.5,856500.0,3597391.5,7889173.0
image_url,3663,3605,https://images-static.nykaa.com/media/catalog/...,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
in_stock,3363,1,True,3363,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mrp,3663.0,NaN,NaN,NaN,757.762217,692.147866,50.0,275.0,578.0,950.0,5375.0
price,3663.0,NaN,NaN,NaN,613.470925,590.075849,41.0,220.0,446.0,763.0,5375.0
product_title,3663,3583,Nykaa Naturals AppleCiderVinegar & Ginger Anti...,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rating,3663.0,NaN,NaN,NaN,4.284848,0.567004,0.0,4.2,4.4,4.5,5.0
rating_count,3663.0,NaN,NaN,NaN,7239.644554,18498.625476,0.0,114.0,781.0,5423.5,180065.0
tags,305,5,NEW,102,NaN,NaN,NaN,NaN,NaN,NaN,NaN
